# NeoProphet starter

Fit a model on the Peyton Manning Wikipedia page views dataset, forecast a year ahead, inspect the components and check accuracy with cross-validation. Then forecast Melbourne pedestrian counts with local temperature as a regressor, leave out closed days, backtest the forecast and build a one-page report and an interactive dashboard.

Run with the **NeoProphet (uv)** kernel, which uses the project `.venv` and the editable `prophet` package from `python/`. The weather sections need network access to [Open-Meteo](https://open-meteo.com).

In [ ]:
import logging
import tempfile
from pathlib import Path

import pandas as pd

from prophet import Prophet, forecast
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

# cmdstanpy resets its level to DEBUG on first use unless a handler already exists
cmdstan_logger = logging.getLogger("cmdstanpy")
cmdstan_logger.addHandler(logging.StreamHandler())
cmdstan_logger.setLevel(logging.WARNING)

## Load data

Prophet expects two columns: `ds` (timestamp) and `y` (value).

In [ ]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples").is_dir())
df = pd.read_csv(ROOT / "examples" / "example_wp_log_peyton_manning.csv", parse_dates=["ds"])
df.head()

## One call

`forecast()` works out the columns and the frequency, fits the model and predicts the horizon. Add `tune=True` to choose the hyperparameters by cross-validation first.

In [ ]:
result = forecast(df, horizon="12 months", country_holidays="US")
print(result.summary())
fig = result.plot()

## Fit and forecast with the full API

The `Prophet` class gives full control; the result is the same model as above without holidays.

In [ ]:
m = Prophet()
m.fit(df)

future = m.make_future_dataframe(periods=365)
fcst = m.predict(future)
fcst[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
fig = m.plot(fcst)

In [ ]:
fig = m.plot_components(fcst)

## Cross-validation

Train on the first two years, then refit every 180 days and score a one-year horizon.

In [ ]:
df_cv = cross_validation(
    m, initial="730 days", period="180 days", horizon="365 days", disable_tqdm=True
)
df_p = performance_metrics(df_cv)
df_p.head()

In [ ]:
fig = plot_cross_validation_metric(df_cv, metric="mape")

## Local temperature as a regressor

Daily pedestrian counts in central Melbourne, cut at the end of 2019 to stay clear of the 2020 lockdowns (`handling_shocks.ipynb` covers those). `weather="Melbourne, Australia"` fetches the daily mean temperature there and adds it as a regressor named `temperature`, for the history and the horizon.

Past dates come from Open-Meteo's archive. For a live series the next 16 days would come from the weather forecast and anything further ahead from climatology (the average for that day of the year over the previous five years). `result.weather.frame` keeps the source of every value. Responses are cached on disk; Open-Meteo's free API is for non-commercial use (see its [terms](https://open-meteo.com/en/terms)).

In [ ]:
ped = pd.read_csv(ROOT / "examples" / "example_pedestrians_covid.csv", parse_dates=["ds"])
ped = ped[ped["ds"] < "2020-01-01"]

walk = forecast(ped, horizon="8 weeks", country_holidays="AU", weather="Melbourne, Australia")
print(walk.summary())
walk.weather.frame.groupby("source").size()

In [ ]:
fig = walk.plot()

How much the temperature moves the forecast, with its fitted coefficient in plain words:

In [ ]:
fig = walk.plot_regressor_effects()

Your own regressors work the same way: `regressors=["promo", "price"]` for columns already in the data, with their future values in `future=`, or a dict of Series covering history and horizon. `regressor_options=` sets the mode and prior scale per regressor. See `seasonality,_holiday_effects,_and_regressors.ipynb` for an example.

## Closed days

Suppose the counter only reports on weekdays. `skip_days="weekends"` leaves Saturdays and Sundays out of training, the future frame and the forecast, so the weekly seasonality is fitted on Monday to Friday only. The charts close the gap, so each Friday joins the following Monday. For sub-daily data, `skip_hours=` does the same for hours of the day.

In [ ]:
weekdays = ped[ped["ds"].dt.dayofweek < 5]

office = forecast(weekdays, horizon="8 weeks", skip_days="weekends", country_holidays="AU")
print(office.summary())
office.forecast.head()

In [ ]:
fig = office.plot()

In [ ]:
fig = office.plot_components()

## Backtest

Hold out the end of the data, refit the same model on the rest and compare the forecast with what actually happened. The holdout defaults to the forecast horizon. `backtest()` runs it once so the report and the dashboard below can reuse it without refitting.

In [ ]:
bt = walk.backtest()
print(bt.metrics)
fig = walk.plot_backtest(backtest=bt)

## One-page report

Forecast, key numbers, components and regressor effects on one A4 page. Save it with `fig.savefig("report.pdf")`.

In [ ]:
fig = walk.report(backtest=bt)

## Interactive dashboard

A self-contained HTML page (needs `plotly`) that works offline. Written to a temporary directory here; open the printed path in a browser.

In [ ]:
path = walk.dashboard(Path(tempfile.mkdtemp()) / "pedestrians.html", backtest=bt)
print(path)